In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/abstracts.csv")
texts = df["text"].tolist()
labels = df["label"].values

In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
X_embed = encoder.encode(texts, show_progress_bar=True)


pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_embed)


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)


X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, labels, test_size=0.25, random_state=42, stratify=labels
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Logistic Regression Baseline

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = LogisticRegression(
    max_iter=2000,
    solver="lbfgs"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Logistic Regression Results")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression Results
              precision    recall  f1-score   support

           0       0.94      0.81      0.87        21
           1       0.83      0.95      0.88        20

    accuracy                           0.88        41
   macro avg       0.89      0.88      0.88        41
weighted avg       0.89      0.88      0.88        41

ROC-AUC: 0.980952380952381


Support Vector Machine (RBF Kernel)

In [5]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score

model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("SVM (RBF) Results")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

SVM (RBF) Results
              precision    recall  f1-score   support

           0       0.94      0.81      0.87        21
           1       0.83      0.95      0.88        20

    accuracy                           0.88        41
   macro avg       0.89      0.88      0.88        41
weighted avg       0.89      0.88      0.88        41

ROC-AUC: 0.9738095238095238


Random Forest (300 Trees)

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Random Forest Results")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest Results
              precision    recall  f1-score   support

           0       0.95      0.86      0.90        21
           1       0.86      0.95      0.90        20

    accuracy                           0.90        41
   macro avg       0.91      0.90      0.90        41
weighted avg       0.91      0.90      0.90        41

ROC-AUC: 0.9833333333333334


XGBoost Baseline (Depth = 6)

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost Results
              precision    recall  f1-score   support

           0       0.95      0.90      0.93        21
           1       0.90      0.95      0.93        20

    accuracy                           0.93        41
   macro avg       0.93      0.93      0.93        41
weighted avg       0.93      0.93      0.93        41

ROC-AUC: 0.9833333333333333


BERT-base Fine-Tuning

In [13]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, ClassLabel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import os
os.environ["WANDB_DISABLED"] = "true"


df = pd.read_csv("abstracts.csv")
dataset = Dataset.from_pandas(df)


dataset = dataset.class_encode_column("label")


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

dataset = dataset.map(tokenize, batched=True)


dataset = dataset.train_test_split(test_size=0.25, stratify_by_column="label")


model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=dataset["train"].features["label"].num_classes
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


args = TrainingArguments(
    output_dir="./bert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=500,
    save_strategy="no"
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


trainer.train()
trainer.evaluate()


Stringifying the column:   0%|          | 0/162 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/162 [00:00<?, ? examples/s]

Map:   0%|          | 0/162 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-2050542336.py:56: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


{'eval_loss': 0.1812802404165268,
 'eval_accuracy': 0.975609756097561,
 'eval_precision': 0.9545454545454546,
 'eval_recall': 1.0,
 'eval_f1': 0.9767441860465116,
 'eval_runtime': 0.6234,
 'eval_samples_per_second': 65.771,
 'eval_steps_per_second': 9.625,
 'epoch': 3.0}

RoBERTa-base Fine-Tuning

In [15]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


os.environ["WANDB_DISABLED"] = "true"


df = pd.read_csv("abstracts.csv")
dataset = Dataset.from_pandas(df)


dataset = dataset.class_encode_column("label")


tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)


dataset = dataset.train_test_split(test_size=0.25, stratify_by_column="label")


model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=dataset["train"].features["label"].num_classes
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


args = TrainingArguments(
    output_dir="./roberta_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=500,
    save_strategy="no"
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


trainer.train()
trainer.evaluate()


Stringifying the column:   0%|          | 0/162 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/162 [00:00<?, ? examples/s]

Map:   0%|          | 0/162 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-968370924.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


{'eval_loss': 0.08313390612602234,
 'eval_accuracy': 0.975609756097561,
 'eval_precision': 1.0,
 'eval_recall': 0.95,
 'eval_f1': 0.9743589743589743,
 'eval_runtime': 0.5729,
 'eval_samples_per_second': 71.571,
 'eval_steps_per_second': 10.474,
 'epoch': 3.0}